In [5]:
pip install tensorflow matplotlib numpy scikit-learn


In [6]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

In [16]:
# Configuration
IMG_HEIGHT = 150
IMG_WIDTH = 150
BATCH_SIZE = 32
EPOCHS = 10
DATASET_DIR = "/content/sample_submission.csv"  # Your dataset folder

In [9]:
# Data Preprocessing
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

In [18]:
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)


NotADirectoryError: [Errno 20] Not a directory: '/content/sample_submission.csv'

In [ ]:
# Build CNN Model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        MaxPooling2D(2,2),

                Conv2D(64, (3,3), activation='relu'),
                    MaxPooling2D(2,2),

                        Conv2D(128, (3,3), activation='relu'),
                            MaxPooling2D(2,2),

                                Flatten(),
                                    Dense(256, activation='relu'),
                                        Dropout(0.5),
                                            Dense(train_generator.num_classes, activation='softmax')
                                            ])

In [ ]:
# Compile Model
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Train Model
history = model.fit(train_generator, validation_data=val_generator, epochs=EPOCHS)

# Save model
model.save("landmark_cnn_model.h5")

# Plot training history
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# Evaluate model
val_generator.reset()
Y_pred = model.predict(val_generator)
y_pred = np.argmax(Y_pred, axis=1)

print("Classification Report:")
print(classification_report(val_generator.classes, y_pred, target_names=list(train_generator.class_indices.keys())))

In [ ]:
def predict_image(img_path, model_path="landmark_cnn_model.h5"):
      model = load_model(model_path)
          img = image.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
              img_array = image.img_to_array(img)
                  img_array = np.expand_dims(img_array, axis=0) / 255.0

                      prediction = model.predict(img_array)
                          class_idx = np.argmax(prediction)
                              class_labels = list(train_generator.class_indices.keys())

                                      return f"Predicted Landmark: {class_labels[class_idx]}"

                                      # Example
                                      print(predict_image("test_images/tajmahal.jpg"))


# New section